# 🔬 Research Paper Explainer Bot — RAG from Scratch
## *With Full Mathematical Derivations & Multilingual Support*

**GitHub:** https://github.com/theqxmlkushal/21_Days_of_LLM_coding/blob/master/Research_Paper_Explainer_Bot.ipynb  
**Stack:** PyPDF2 · SentenceTransformers · FAISS · Groq (Llama 3) · Streamlit · langdetect

> **Goal:** Build every step of a RAG pipeline from scratch — no LangChain, no black boxes.  
> Every equation below has a corresponding code cell so you can trace math → implementation.

---


## 📋 Table of Contents

1. [Setup & Imports](#1-setup--imports)
2. [Step 1: PDF Parsing](#2-step-1-pdf-parsing)
3. [Step 2: Sliding-Window Chunking — Math](#3-step-2-sliding-window-chunking)
4. [Step 3: Dense Embeddings & Multilingual Support — Math](#4-step-3-dense-embeddings)
5. [Step 4: FAISS Vector Index & Cosine Similarity — Math](#5-step-4-faiss-vector-index)
6. [Step 5: RAG Query & LLM Generation — Math](#6-step-5-rag-query--llm-generation)
7. [Step 6: Multilingual Cross-Lingual Retrieval — Math](#7-step-6-multilingual-retrieval)
8. [Full Pipeline End-to-End Demo](#8-full-pipeline-demo)
9. [Hyperparameter Analysis](#9-hyperparameter-analysis)
10. [References](#10-references)


---
## 1. Setup & Imports

Install all dependencies. The `--quiet` flag suppresses verbose pip output.


In [ ]:
# Install dependencies
!pip install PyPDF2 sentence-transformers faiss-cpu groq langdetect plotly --quiet

In [ ]:
import PyPDF2
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import re
import json
import time
from io import BytesIO
from typing import List, Dict, Tuple
from groq import Groq
import plotly.graph_objects as go
import plotly.express as px
from collections import Counter

# Optional: langdetect for language identification
try:
    from langdetect import detect, detect_langs
    print("✅ langdetect available")
except ImportError:
    print("⚠️  langdetect not installed — run: pip install langdetect")

print("✅ All imports successful")
print(f"   FAISS version  : {faiss.__version__}")
print(f"   NumPy version  : {np.__version__}")


---
## 2. Step 1: PDF Parsing

### What we do
Read a PDF file and extract all text from every page using **PyPDF2**.

### Why it's non-trivial
PDFs store text in a page-description format, not as plain text.  
PyPDF2 reconstructs the reading order from the PDF's internal structure.

### Text cleaning
After extraction we normalise whitespace:

$$\text{cleaned} = \texttt{re.sub}(\texttt{r"\\s+"},\ \texttt{" "},\ \text{raw\_text}).\texttt{strip()}$$


In [ ]:
def load_pdf(pdf_path: str) -> str:
    """
    Load a PDF from disk and return its full text as a single clean string.
    
    Args:
        pdf_path: Path to the PDF file.
    Returns:
        Cleaned text string.
    """
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        pages = []
        for i, page in enumerate(reader.pages):
            t = page.extract_text()
            if t:
                pages.append(t)
                print(f"  Page {i+1}: {len(t):>6} chars extracted")
    
    raw   = " ".join(pages)
    clean = re.sub(r"\s+", " ", raw).strip()
    
    print(f"\n📄 Total raw chars     : {len(raw):,}")
    print(f"📄 After normalisation : {len(clean):,}")
    print(f"📄 Pages processed     : {len(pages)}")
    return clean


# ── DEMO: show first 500 chars of any PDF you upload ──
# Replace with your own PDF path in Colab:
#   from google.colab import files
#   uploaded = files.upload()
#   pdf_text = load_pdf(list(uploaded.keys())[0])

# For now we create a synthetic sample:
SAMPLE_TEXT = (
    "This paper presents a novel approach to information retrieval using "
    "Retrieval-Augmented Generation (RAG). We propose combining dense embeddings "
    "with a large language model to answer questions about scientific documents. "
    "Our method outperforms BM25 baselines by 12 points on the NaturalQuestions "
    "benchmark. The key innovation is a sliding-window chunking strategy with "
    "50-token overlap, which preserves cross-sentence context that fixed-window "
    "approaches miss. Experiments on three datasets confirm the effectiveness. " * 20
)
print(f"\nSample text length: {len(SAMPLE_TEXT):,} chars")
print("Preview:", SAMPLE_TEXT[:200], "…")


---
## 3. Step 2: Sliding-Window Chunking

### The Problem
LLMs and embedding models have a **context window limit** (e.g., 512 tokens ≈ ~2,000 chars).  
We cannot feed the entire paper. We must split it into manageable pieces.

### The Formula
Given text $T$ of length $|T|$, chunk size $S$, and overlap $O$:

$$\text{chunk}_i = T\bigl[\, i(S - O) \;:\; i(S - O) + S \,\bigr]$$

Number of chunks produced:

$$N = \left\lceil \frac{|T| - O}{S - O} \right\rceil$$

### Example with S = 500, O = 50

| Chunk | Start | End | Overlap with prev |
|-------|-------|-----|-------------------|
| 0 | 0 | 500 | — |
| 1 | 450 | 950 | chars 450–500 |
| 2 | 900 | 1400 | chars 900–950 |
| i | $i \cdot 450$ | $i \cdot 450 + 500$ | 50 chars |

### Why Overlap?
Without overlap, a sentence could be split across two chunks, losing its meaning.  
With overlap = 50, every chunk shares 50 characters with its neighbour —  
ensuring no sentence is permanently severed.


In [ ]:
def chunk_text(text: str, size: int = 500, overlap: int = 50) -> List[str]:
    """
    Sliding-window chunker.
    
    Formula: chunk_i = text[i*(size-overlap) : i*(size-overlap)+size]
    
    Args:
        text:    Input string.
        size:    Window size in characters.
        overlap: Overlap between consecutive windows.
    Returns:
        List of non-empty chunk strings.
    """
    assert overlap < size, "Overlap must be smaller than chunk size"
    step   = size - overlap   # stride between chunk starts
    chunks = []
    start  = 0
    while start < len(text):
        chunk = text[start : start + size].strip()
        if chunk:
            chunks.append(chunk)
        start += step
    return chunks


# ── Compute expected chunk count analytically ──
def expected_chunks(text_len: int, size: int, overlap: int) -> int:
    import math
    return math.ceil((text_len - overlap) / (size - overlap))


# ── Demo ──
chunks = chunk_text(SAMPLE_TEXT, size=500, overlap=50)

print(f"Text length  : {len(SAMPLE_TEXT):,} chars")
print(f"Chunk size   : 500 chars")
print(f"Overlap      : 50 chars")
print(f"Chunks (actual)   : {len(chunks)}")
print(f"Chunks (formula)  : {expected_chunks(len(SAMPLE_TEXT), 500, 50)}")
print()

# Verify overlap
for i in range(min(3, len(chunks)-1)):
    shared = len(set(chunks[i][-50:]) & set(chunks[i+1][:50]))
    print(f"Chunk {i} tail: …{chunks[i][-30:]!r}")
    print(f"Chunk {i+1} head: {chunks[i+1][:30]!r}…")
    print(f"Overlap (chars): {50}  |  unique overlap chars: {shared}")
    print("─" * 60)

print(f"\nAvg chunk length : {sum(len(c) for c in chunks)/len(chunks):.0f} chars")
print(f"Min chunk length : {min(len(c) for c in chunks)}")
print(f"Max chunk length : {max(len(c) for c in chunks)}")


In [ ]:
# ── Visualise chunk length distribution ──
import plotly.graph_objects as go

lengths = [len(c) for c in chunks]
fig = go.Figure(go.Histogram(
    x=lengths,
    nbinsx=20,
    marker_color="#7B6FFF",
    opacity=0.85,
    name="Chunk lengths",
))
fig.update_layout(
    title="Distribution of Chunk Lengths",
    xaxis_title="Characters per chunk",
    yaxis_title="Count",
    plot_bgcolor="#0D0D14",
    paper_bgcolor="#0D0D14",
    font=dict(color="#C0C0D0"),
    bargap=0.05,
)
fig.show()
print(f"Chunks within 10% of target size: {sum(1 for l in lengths if abs(l-500)/500 < 0.1)}/{len(lengths)}")


---
## 4. Step 3: Dense Embeddings & Multilingual Support

### What Is an Embedding?
An embedding function $E$ maps variable-length text to a fixed-size dense vector:

$$E : \Sigma^* \rightarrow \mathbb{R}^d \quad (d = 384 \text{ for MiniLM},\ 768 \text{ for MPNet})$$

### Sentence-BERT Architecture
The model uses a **siamese Transformer network** trained on sentence pairs:

$$E(\text{sentence}) = \text{mean\_pool}\bigl(\text{BERT}(\text{tokenize}(\text{sentence}))\bigr)$$

Mean pooling averages token embeddings across sequence length $L$:

$$\mathbf{e} = \frac{1}{L} \sum_{i=1}^{L} \mathbf{h}_i \quad \in \mathbb{R}^d$$

where $\mathbf{h}_i$ is the BERT hidden state at position $i$.

### Multilingual Model
`paraphrase-multilingual-MiniLM-L12-v2` is trained on **parallel corpora** in 50+ languages.  
This forces the model to place semantically equivalent phrases from different languages  
*close together* in $\mathbb{R}^d$:

$$\|E(\text{"cat"}) - E(\text{"gato"})\| \ll \|E(\text{"cat"}) - E(\text{"dog"})\|$$

$$E(\text{"cat"}) \approx E(\text{"gato"}) \approx E(\text{"猫"}) \approx E(\text{"बिल्ली"})$$

### L2 Normalisation (Required for Cosine Similarity)
We normalise all embeddings before storing them:

$$\hat{\mathbf{e}} = \frac{\mathbf{e}}{\|\mathbf{e}\|_2}$$

After normalisation: $\|\hat{\mathbf{e}}\|_2 = 1$, so $\hat{\mathbf{q}} \cdot \hat{\mathbf{c}} = \cos(\theta_{qc})$


In [ ]:
# Load the multilingual embedding model
print("Loading multilingual embedding model…")
t0 = time.time()

embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print(f"Model loaded in {time.time()-t0:.2f}s")
print(f"Embedding dimension : {embed_model.get_sentence_embedding_dimension()}")
print(f"Max sequence length : {embed_model.max_seq_length}")


In [ ]:
# Embed all chunks (with L2 normalisation for cosine sim)
print("Encoding chunks…")
t0 = time.time()

embeddings = embed_model.encode(
    chunks,
    show_progress_bar=True,
    normalize_embeddings=True,    # ← L2 normalise so ||e|| = 1
    batch_size=32,
)

t1 = time.time()
print(f"\n✅ Embeddings shape  : {embeddings.shape}")
print(f"   Encoding time     : {t1-t0:.2f}s")
print(f"   Throughput        : {len(chunks)/(t1-t0):.1f} chunks/sec")
print(f"   Memory (float32)  : {embeddings.nbytes / 1024:.1f} KB")

# Verify normalisation
norms = np.linalg.norm(embeddings, axis=1)
print(f"\nNorm stats (should all be ≈ 1.0):")
print(f"  Mean : {norms.mean():.6f}")
print(f"  Min  : {norms.min():.6f}")
print(f"  Max  : {norms.max():.6f}")


In [ ]:
# ── Cross-lingual similarity demo ──
test_pairs = [
    ("attention mechanism", "mécanisme d'attention"),
    ("neural network",      "ニューラルネットワーク"),
    ("machine learning",    "Maschinelles Lernen"),
    ("deep learning",       "اَلتَّعَلُّم العَمِيق"),
    ("research paper",      "论文"),
]

print("Cross-Lingual Semantic Similarity (multilingual model)\n")
print(f"{'English':<25} {'Foreign':<30} {'Cosine Sim':>12}")
print("─" * 70)

for en, foreign in test_pairs:
    e_en = embed_model.encode([en],      normalize_embeddings=True)[0]
    e_fo = embed_model.encode([foreign], normalize_embeddings=True)[0]
    sim  = float(np.dot(e_en, e_fo))
    bar  = "█" * int(sim * 20)
    print(f"{en:<25} {foreign:<30} {sim:>8.4f}  {bar}")

print("\n→ Similarity > 0.6 confirms semantic alignment across languages.")


---
## 5. Step 4: FAISS Vector Index & Cosine Similarity

### Why FAISS?
Naïve nearest-neighbour search is $O(N \cdot d)$ per query — fine for small corpora.  
FAISS (Facebook AI Similarity Search) provides highly optimised exact and approximate search.

### Cosine Similarity (v2 upgrade from L2)
Given normalised vectors $\hat{\mathbf{q}}$ (query) and $\hat{\mathbf{c}}_i$ (chunk $i$):

$$\cos(\hat{\mathbf{q}}, \hat{\mathbf{c}}_i) = \hat{\mathbf{q}} \cdot \hat{\mathbf{c}}_i = \sum_{j=1}^{d} \hat{q}_j \cdot \hat{c}_{ij} \quad \in [-1,\, 1]$$

For unit vectors, inner product **equals** cosine similarity.  
FAISS `IndexFlatIP` computes inner products — so with normalised vectors we get exact cosine sim.

### Top-K Retrieval
$$\text{Top-K} = \underset{i \in \{1,\ldots,N\}}{\operatorname{arg\,max}_K} \;\cos(\hat{\mathbf{q}},\, \hat{\mathbf{c}}_i)$$

### L2 vs Cosine — Which is Better?

| Metric | Formula | Range | Best for |
|--------|---------|-------|----------|
| L2 Distance | $\sqrt{\sum_i (q_i - c_i)^2}$ | $[0, \infty)$ | Absolute distance matters |
| Cosine Similarity | $\hat{\mathbf{q}} \cdot \hat{\mathbf{c}}$ | $[-1, 1]$ | **Semantic direction** (RAG ✓) |

Cosine similarity is **scale-invariant**: a short chunk and a long chunk covering the same topic  
will score equally, while L2 penalises the shorter one.


In [ ]:
def build_index(embeddings: np.ndarray) -> faiss.IndexFlatIP:
    """
    Build a FAISS flat inner-product index.
    With L2-normalised vectors, inner product = cosine similarity.
    
    Time complexity  : O(1) to build, O(N·d) per query (exact search)
    Space complexity : O(N·d) for the stored vectors
    """
    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)         # Inner Product (= cosine for unit vecs)
    index.add(embeddings.astype("float32"))
    print(f"✅ FAISS index built")
    print(f"   Type         : IndexFlatIP (exact cosine)")
    print(f"   Dimension    : {dim}")
    print(f"   Stored vecs  : {index.ntotal}")
    print(f"   Memory est.  : {index.ntotal * dim * 4 / 1024:.1f} KB")
    return index


index = build_index(embeddings)


In [ ]:
def retrieve(query: str, embed_model, index: faiss.IndexFlatIP,
             chunks: List[str], top_k: int = 3) -> List[Dict]:
    """
    Retrieve top-k most similar chunks using cosine similarity.
    
    Pipeline:
      1. Encode query → normalised embedding q̂
      2. FAISS inner product search → top-k indices + scores
      3. Package results with metadata
    """
    # Step 1: embed and normalise query
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    
    # Step 2: search index
    scores, indices = index.search(q_emb, top_k)
    
    # Step 3: package results
    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0])):
        if idx < 0:
            continue
        results.append({
            "rank":       rank + 1,
            "chunk_id":   int(idx),
            "text":       chunks[idx],
            "cosine_sim": float(score),
            "length":     len(chunks[idx]),
        })
    return results


# ── Demo retrieval ──
query = "What is the main contribution of this paper?"
results = retrieve(query, embed_model, index, chunks, top_k=3)

print(f"Query: '{query}'\n")
print(f"{'Rank':<6} {'Chunk ID':<10} {'Cosine Sim':>12} {'Length':>8}")
print("─" * 40)
for r in results:
    bar = "█" * int(r['cosine_sim'] * 30)
    print(f"{r['rank']:<6} #{r['chunk_id']+1:<9} {r['cosine_sim']:>12.4f} {r['length']:>8}")
    print(f"         {bar}")
    print()


In [ ]:
# ── Visualise similarity matrix (chunk × chunk) ──
# Compute pairwise cosine similarities for first 20 chunks
n_show = min(20, len(embeddings))
sub    = embeddings[:n_show].astype("float32")
sim_matrix = sub @ sub.T   # all dot products → cosine sim (unit vecs)

fig = go.Figure(go.Heatmap(
    z=sim_matrix,
    colorscale="Viridis",
    zmin=0, zmax=1,
    colorbar=dict(title="Cosine<br>Similarity"),
))
fig.update_layout(
    title=f"Pairwise Cosine Similarity — First {n_show} Chunks",
    xaxis_title="Chunk index",
    yaxis_title="Chunk index",
    plot_bgcolor="#0D0D14",
    paper_bgcolor="#0D0D14",
    font=dict(color="#C0C0D0"),
    height=500,
)
fig.show()
print("→ Bright cells = semantically similar chunks (likely discuss the same concept).")


---
## 6. Step 5: RAG Query & LLM Generation

### The RAG Probability Model
RAG conditions the language model on retrieved context:

$$P(\text{answer} \mid \text{query}) \;=\; \sum_{z \in \mathcal{Z}} P(\text{answer} \mid \text{query},\, z) \cdot P(z \mid \text{query})$$

where $\mathcal{Z}$ is the set of all possible context documents.  
In practice we approximate by taking the top-K chunks:

$$\hat{P}(\text{answer} \mid \text{query}) \approx P\bigl(\text{answer} \mid \text{query},\, \text{chunk}_{(1)},\ldots,\text{chunk}_{(K)}\bigr)$$

### Temperature Sampling
The LLM's next-token probabilities are modulated by temperature $\tau$:

$$P(w_t = w \mid w_{<t}) = \frac{\exp(z_w / \tau)}{\sum_{w'} \exp(z_{w'} / \tau)}$$

- $\tau \to 0$: deterministic (argmax decoding)  
- $\tau = 1$: standard softmax (raw logits)  
- $\tau > 1$: flatter distribution → more creative, less coherent

### Prompt Template

```
SYSTEM: You are a multilingual research assistant.
        Answer ONLY from the context provided.
        Detect the question language and reply in the same language.

USER:   Paper language: {lang}
        Context:
          [Chunk 1]: {chunk_text_1}
          [Chunk 2]: {chunk_text_2}
          ...
          [Chunk K]: {chunk_text_K}
        Question: {query}
        Provide a structured answer:
```


In [ ]:
# ── Set your Groq API key ──
import os

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
# In Colab: use the Secrets feature (🔑 icon on the left panel)
# from google.colab import userdata
# GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    print("⚠️  Set GROQ_API_KEY to test generation. Retrieval steps above work without it.")
else:
    print("✅ Groq API key found")


In [ ]:
def generate_answer(
    query: str,
    chunks: List[Dict],
    groq_client: Groq,
    model: str = "llama-3.1-8b-instant",
    paper_lang: str = "English",
    answer_lang: str = "auto",
    temperature: float = 0.7,
    max_tokens: int = 600,
) -> str:
    """
    RAG generation step: inject retrieved chunks into LLM prompt.
    
    Args:
        query      : User question (any language).
        chunks     : Retrieved chunks from retrieve().
        groq_client: Initialised Groq client.
        model      : Groq-hosted model name.
        paper_lang : Detected paper language (for prompt context).
        answer_lang: Target answer language ('auto' = match question).
        temperature: Sampling temperature τ ∈ [0, 1].
        max_tokens : Maximum tokens in generated answer.
    Returns:
        Generated answer string.
    """
    # Build context block
    context = "\n\n".join(
        f"[Chunk {c['chunk_id']+1}]:\n{c['text']}" for c in chunks
    )
    
    # Language instruction
    if answer_lang == "auto":
        lang_inst = (
            "Detect the language of the question and respond in that SAME language. "
            f"The paper is in {paper_lang}. You may quote the paper in its original language."
        )
    else:
        lang_inst = f"Always respond in {answer_lang}, regardless of the question language."
    
    system_prompt = (
        "You are an expert multilingual research assistant. "
        "Answer questions using ONLY the context provided. "
        "Cite chunk numbers when relevant (e.g., 'As stated in Chunk 3…'). "
        "If the context lacks information, say so clearly. "
        + lang_inst
    )
    
    user_prompt = (
        f"Research paper language: {paper_lang}\n\n"
        f"Context from paper:\n\n{context}\n\n"
        f"Question: {query}\n\n"
        "Provide a clear, structured answer:"
    )
    
    response = groq_client.chat.completions.create(
        model       = model,
        messages    = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        max_tokens  = max_tokens,
        temperature = temperature,
    )
    return response.choices[0].message.content


# ── Test if key is available ──
if GROQ_API_KEY:
    client  = Groq(api_key=GROQ_API_KEY)
    q       = "What is the main contribution of this paper?"
    results = retrieve(q, embed_model, index, chunks, top_k=3)
    answer  = generate_answer(q, results, client)
    print(f"Q: {q}\n")
    print(f"A: {answer}")
else:
    print("⏭️  Skipping generation test — no API key set.")
    print("   Set GROQ_API_KEY and re-run this cell.")


---
## 7. Step 6: Multilingual & Cross-Lingual Retrieval

### How Multilingual Embeddings Work

The model is trained on **parallel corpora** — datasets of the same sentence in 50+ languages.  
Training forces the encoder to produce similar vectors for translations.

The training objective (contrastive loss on sentence pairs):

$$\mathcal{L} = -\log \frac{\exp\bigl(\cos(E(s),\, E(s^+)) / \tau_c\bigr)}{\sum_{j} \exp\bigl(\cos(E(s),\, E(s_j^-)) / \tau_c\bigr)}$$

where $s$ is the anchor sentence, $s^+$ is a positive (translation or paraphrase),  
$s_j^-$ are negative examples, and $\tau_c$ is the contrastive temperature.

### Cross-Lingual Search
Because translations land close in $\mathbb{R}^d$:

$$\cos\bigl(E(\text{"attention mechanism"}),\; E(\text{"mécanisme d'attention"})\bigr) \approx 0.95$$

A Hindi query can retrieve relevant English paper chunks because:

$$\cos\bigl(E(\text{Hindi query}),\; E(\text{English chunk})\bigr) > \cos\bigl(E(\text{Hindi query}),\; E(\text{unrelated English chunk})\bigr)$$

### Language Detection
We use `langdetect` (based on Google's language detection library).  
It models text as character $n$-gram profiles and returns a probability distribution:

$$P(\text{language} = l \mid \text{text}) \propto \text{ngram-profile-similarity}(\text{text},\, \text{profile}_l)$$


In [ ]:
# ── Language detection demo ──
try:
    from langdetect import detect, detect_langs

    test_texts = {
        "English":  "This paper presents a novel method for information retrieval.",
        "Hindi":    "यह पेपर सूचना पुनर्प्राप्ति के लिए एक नई विधि प्रस्तुत करता है।",
        "French":   "Cet article présente une nouvelle méthode pour la récupération d'information.",
        "Chinese":  "本文提出了一种新的信息检索方法。",
        "Arabic":   "تقدم هذه الورقة طريقة جديدة لاسترجاع المعلومات.",
        "German":   "Dieses Paper präsentiert eine neue Methode zur Informationsgewinnung.",
        "Japanese": "この論文は、情報検索のための新しい方法を提示します。",
    }

    print(f"{'Actual Language':<15} {'Detected':>12} {'Confidence':>12} {'Correct?':>10}")
    print("─" * 52)
    for lang, text in test_texts.items():
        try:
            detected = detect_langs(text)
            top = detected[0]
            code = top.lang
            conf = top.prob
            correct = "✅" if lang.lower()[:2] == code[:2] else "❌"
            print(f"{lang:<15} {code:>12} {conf:>12.3f} {correct:>10}")
        except Exception as e:
            print(f"{lang:<15} {'ERROR':>12} {str(e)[:30]}")
except ImportError:
    print("langdetect not available — run: pip install langdetect")


In [ ]:
# ── Cross-lingual retrieval experiment ──
QUERIES_MULTILANG = [
    ("What is the main contribution?",          "English"),
    ("इस पेपर का मुख्य योगदान क्या है?",          "Hindi"),
    ("Quelle est la principale contribution?", "French"),
    ("これは何を貢献していますか？",                  "Japanese"),
    ("Was ist der Hauptbeitrag?",              "German"),
]

print("Cross-Lingual Retrieval — same paper, different query languages\n")
print("All queries ask about 'main contribution' in different languages.")
print("="*65)

for query, lang in QUERIES_MULTILANG:
    results = retrieve(query, embed_model, index, chunks, top_k=1)
    if results:
        top = results[0]
        print(f"\n[{lang}] Query: {query[:60]}")
        print(f"  → Retrieved Chunk #{top['chunk_id']+1}  |  Cosine: {top['cosine_sim']:.4f}")
        print(f"  → Preview: {top['text'][:100]}…")
    else:
        print(f"\n[{lang}] No results.")

print("\n→ Notice: different-language queries retrieve the SAME relevant chunk.")
print("  This is cross-lingual retrieval working correctly.")


---
## 8. Full Pipeline End-to-End Demo

Putting all steps together into a clean class — exactly what `streamlit_app.py` uses.


In [ ]:
class ResearchPaperRAG:
    """
    Full RAG pipeline from scratch.
    
    Steps:
      1. PDF → Text        (PyPDF2)
      2. Text → Chunks     (sliding-window)
      3. Chunks → Vectors  (multilingual SentenceBERT + L2 norm)
      4. Vectors → Index   (FAISS IndexFlatIP = cosine sim)
      5. Query → Chunks    (cosine retrieval)
      6. Chunks → Answer   (Groq LLM)
    """

    def __init__(self, text: str, config: Dict, groq_api_key: str = ""):
        self.config = config
        self.client = Groq(api_key=groq_api_key) if groq_api_key else None

        # Pipeline
        self.text        = text
        self.chunks      = chunk_text(text, config["chunk_size"], config["chunk_overlap"])
        self.embed_model = SentenceTransformer(config["embedding_model"])
        self.embeddings  = self.embed_model.encode(
            self.chunks, normalize_embeddings=True, show_progress_bar=False
        )
        self.index       = build_index(self.embeddings)

    def retrieve(self, query: str) -> List[Dict]:
        return retrieve(query, self.embed_model, self.index, self.chunks, self.config["top_k"])

    def ask(self, query: str) -> Dict:
        chunks = self.retrieve(query)
        if not self.client:
            return {"answer": "No API key provided.", "chunks": chunks, "success": False}
        answer = generate_answer(query, chunks, self.client,
                                  model       = self.config["llm_model"],
                                  temperature = self.config["temperature"],
                                  max_tokens  = self.config["max_tokens"])
        return {"answer": answer, "chunks": chunks, "success": True}

    def stats(self) -> Dict:
        return {
            "n_chunks":  len(self.chunks),
            "n_chars":   len(self.text),
            "emb_dim":   self.embeddings.shape[1],
            "emb_model": self.config["embedding_model"],
        }


# ── Initialise with default config ──
CONFIG = {
    "chunk_size":      500,
    "chunk_overlap":   50,
    "embedding_model": "paraphrase-multilingual-MiniLM-L12-v2",
    "top_k":           3,
    "llm_model":       "llama-3.1-8b-instant",
    "temperature":     0.7,
    "max_tokens":      600,
}

print("Initialising RAG pipeline…")
rag = ResearchPaperRAG(SAMPLE_TEXT, CONFIG, groq_api_key=GROQ_API_KEY)
s   = rag.stats()
print(f"\nPipeline ready!")
print(f"  Chunks         : {s['n_chunks']}")
print(f"  Chars          : {s['n_chars']:,}")
print(f"  Embedding dim  : {s['emb_dim']}")
print(f"  Model          : {s['emb_model']}")


In [ ]:
# ── Run a question ──
q = "What is the main contribution of this paper?"
print(f"Q: {q}\n")

if GROQ_API_KEY:
    result = rag.ask(q)
    print(f"A: {result['answer']}\n")
    print("Retrieved chunks:")
    for c in result["chunks"]:
        print(f"  Rank {c['rank']} · Chunk #{c['chunk_id']+1} · Cosine {c['cosine_sim']:.4f}")
        print(f"  {c['text'][:120]}…\n")
else:
    # Just show retrieval
    result = rag.retrieve(q)
    print("(Generation skipped — no API key)\n")
    print("Retrieved chunks:")
    for c in result:
        print(f"  Rank {c['rank']} · Chunk #{c['chunk_id']+1} · Cosine {c['cosine_sim']:.4f}")
        print(f"  {c['text'][:120]}…\n")


---
## 9. Hyperparameter Analysis

### Effect of Chunk Size on Retrieval Quality

**Trade-off:** Larger chunks contain more context but become noisier.  
Smaller chunks are more precise but may miss cross-sentence context.

**Optimal range** for most academic papers: $S \in [300, 700]$ characters.

### Effect of Top-K on Answer Quality

$K$ controls the context window size fed to the LLM:

$$\text{context\_tokens} \approx K \cdot \frac{S}{4} \quad (\approx 4 \text{ chars/token})$$

| K | Context tokens | Quality | Cost |
|---|---------------|---------|------|
| 1 | ~125 | Precise but may miss info | Low |
| 3 | ~375 | **Balanced (recommended)** | Medium |
| 5 | ~625 | Rich context | Higher |
| 10 | ~1250 | May introduce noise | Highest |


In [ ]:
# ── Hyperparameter sweep: chunk size vs retrieval spread ──
test_query = "main contribution of this paper"
chunk_sizes = [200, 300, 500, 700, 1000]

print(f"Query: '{test_query}'\n")
print(f"{'Chunk Size':>12} {'N Chunks':>10} {'Avg Sim':>10} {'Top Sim':>10} {'Sim Range':>12}")
print("─" * 58)

results_sweep = []
for cs in chunk_sizes:
    c_list  = chunk_text(SAMPLE_TEXT, size=cs, overlap=cs//10)
    e_list  = embed_model.encode(c_list, normalize_embeddings=True, show_progress_bar=False)
    idx_tmp = build_index(e_list)
    res     = retrieve(test_query, embed_model, idx_tmp, c_list, top_k=3)
    sims    = [r["cosine_sim"] for r in res]
    results_sweep.append({
        "chunk_size": cs, "n_chunks": len(c_list),
        "avg_sim": sum(sims)/len(sims), "top_sim": sims[0],
        "sim_range": sims[0] - sims[-1],
    })
    print(f"{cs:>12} {len(c_list):>10} {sims[0] if sims else 0:>10.4f} {max(sims) if sims else 0:>10.4f} {(sims[0]-sims[-1]) if len(sims)>1 else 0:>12.4f}")

# Rebuild original index with default config
embeddings = embed_model.encode(chunks, normalize_embeddings=True, show_progress_bar=False)
index      = build_index(embeddings)
print("\n(Original index restored)")


In [ ]:
# ── Visualise the sweep ──
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=[r["chunk_size"] for r in results_sweep],
    y=[r["top_sim"] for r in results_sweep],
    mode="lines+markers",
    name="Top-1 Similarity",
    line=dict(color="#7B6FFF", width=2),
    marker=dict(size=10),
))
fig.add_trace(go.Scatter(
    x=[r["chunk_size"] for r in results_sweep],
    y=[r["avg_sim"] for r in results_sweep],
    mode="lines+markers",
    name="Avg Similarity (top-3)",
    line=dict(color="#38BDF8", width=2, dash="dot"),
    marker=dict(size=8),
))
fig.update_layout(
    title="Chunk Size vs Retrieval Similarity",
    xaxis_title="Chunk Size (chars)",
    yaxis_title="Cosine Similarity",
    yaxis=dict(range=[0, 1.05]),
    plot_bgcolor="#0D0D14", paper_bgcolor="#0D0D14",
    font=dict(color="#C0C0D0"), height=380,
    legend=dict(font=dict(color="#C0C0D0")),
)
fig.show()
print("→ Larger chunks often score higher (more content matches the query).")
print("  But very large chunks can hurt precision (too much irrelevant content).")


---
## 10. References

| # | Paper | Authors | Venue | Year |
|---|-------|---------|-------|------|
| [1] | **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks** | Lewis et al. | NeurIPS | 2020 |
| [2] | **Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks** | Reimers & Gurevych | EMNLP | 2019 |
| [3] | **Billion-scale similarity search with GPUs (FAISS)** | Johnson et al. | IEEE Trans. | 2021 |
| [4] | **Llama 3: Open Foundation and Fine-Tuned Chat Models** | Meta AI | — | 2024 |
| [5] | **Dense Passage Retrieval for Open-Domain Question Answering** | Karpukhin et al. | EMNLP | 2020 |
| [6] | **Making Monolingual Sentence Embeddings Multilingual** | Reimers & Gurevych | EMNLP | 2020 |
| [7] | **BERT: Pre-training of Deep Bidirectional Transformers** | Devlin et al. | NAACL | 2019 |
| [8] | **Attention Is All You Need** | Vaswani et al. | NeurIPS | 2017 |

---

## 📋 Project Template

**Project Title:** Research Paper Explainer Bot — Multilingual RAG from Scratch

**GitHub Repository:** https://github.com/theqxmlkushal/21_Days_of_LLM_coding/blob/master/Research_Paper_Explainer_Bot.ipynb

**Technologies / Concepts Used:**
- Retrieval-Augmented Generation (RAG) pipeline from scratch
- FAISS (IndexFlatIP) with cosine similarity search
- SentenceTransformers — multilingual dense embeddings (50+ languages)
- Groq API (Llama 3.1, Mixtral) for fast LLM inference
- Streamlit for the web application
- langdetect for automatic language identification
- PyPDF2 for PDF text extraction
- Plotly for interactive charts

**Project Overview:**
A production-ready web app that lets users chat with any research paper PDF. The entire RAG pipeline is built from scratch with no LangChain or LlamaIndex, making every step transparent and educational. v2.0 adds multilingual support (auto-detects paper language, supports cross-lingual Q&A in 50+ languages), a citation extractor, keyword analysis, and upgraded cosine similarity retrieval.

**Team Member Roles & Contribution Distribution:**

| Role | Responsibility | Contribution |
|------|----------------|-------------|
| ML Engineer | RAG pipeline, embeddings, FAISS, multilingual support | 35% |
| Backend Developer | Groq API, prompt engineering, language detection | 25% |
| Frontend Developer | Streamlit UI, CSS, Plotly charts, tab design | 25% |
| Research & Docs | Math derivations, README, notebook, references | 15% |

**Research Papers / References Related to the Topic:**
See the References table above (items [1]–[8]).
